STEP 1

The Cheat Sheet (Requirement Gathering Principles)

    1) Focus on the problem, not the visuals: Never start a conversation by asking, "What charts do you want to see?". Always ask: "What business problem are you currently facing?".

    2) Define operational expectations: Once the problem is defined, ask what exactly the user expects from the tool and what specific action they will take when they see the data.

    3) Propose an initial solution (Prototyping): Don't build the final report blindly. Sketch an initial layout (even on paper) and make sure it actually solves the business need.

    4) Define the timeframe and frequency: Establish the time context – are decisions made daily, or strategically once a quarter?

    5) Force specific metrics (Prioritization): Limit the number of key indicators to 3-5, instead of agreeing to "show all the data on one screen".

The Interrogation (Stakeholder Questions)
For the Director (Strategic Perspective):

    1. What are the 3 most important strategic goals for the company this year?
    2. By looking at which specific KPIs will you know that we are heading in the right direction and succeeding?
    3. What recent business decision did you have to make blindly because you lacked the right data?
    4. During which meetings will this report be used and presented?

For the Operations Manager (Tactical/Operational Perspective):

    5. What specific problem are you dealing with, and which process generates the most customer complaints?
    6. What exactly do you need to see on your screen every morning to catch a problem before it escalates?
    7. How do you currently track these operations, and what annoys you the most about this process?
    8. When you see a metric deteriorating, what is your first action, and who do you call?

Why is it dangerous to put "all metrics on one screen"?

It is simply unnecessary and causes the data to blend together into visual noise. A dashboard shouldn't just exist for the sake of passively checking numbers. Its primary purpose is to deliver tangible value and drive beneficial business outcomes. Cluttering the screen prevents users from extracting that value.

What is the difference between a Metric and an Actionable Insight?

A metric is just a standalone "number" that we see on a dashboard. An actionable insight, on the other hand, is ready-to-use information deduced from those metrics and KPIs, designed specifically to trigger a further action that drives business development or solves a problem.

STEP 2

1. Executive Summary

The goal of the BI solution is to provide two complementary reporting experiences:

- a strategic dashboard for the Executive Director to monitor sales performance, growth, geographical performance and customer value,
- an operational dashboard for the Operations Manager to identify delivery bottlenecks, delayed orders and logistics anomalies requiring action.


2. Target Audience & User Personas

Executive Director (Strategic):
The Executive Director uses the report to monitor overall business performance, revenue growth, geographical performance and customer value. The dashboard should help identify high-performing areas, underperforming markets and customer segments that require strategic attention.

Operations Manager (Tactical):
The Operations Manager uses the report for day-to-day delivery monitoring. The dashboard should help identify delayed orders, delivery bottlenecks, unusually long delivery times and states where operational intervention may be required.


3. Key Metrics & KPIs

- Total Revenue
Definition: Total customer charge generated by sales, including product price and freight.
Formula:
Total Revenue = SUMX(fact_sales, price + freight_value)

- Total Orders
Definition: Number of unique customer orders.
Formula:
Total Orders = DISTINCTCOUNT(fact_sales[order_id])

- Average Order Value (AOV)
Definition: Average revenue generated per unique order.
Formula:
AOV = Total Revenue / Total Orders

- Revenue Last Year
Definition: Revenue for the equivalent selected period in the previous year.
Formula:
Revenue Last Year =
CALCULATE(
    [Total Revenue],
    SAMEPERIODLASTYEAR(dim_date[full_date])
)

- Revenue YoY %
Definition: Percentage revenue change compared with the same period in the previous year.
Formula:
Revenue YoY % =
(Total Revenue - Revenue Last Year) / Revenue Last Year

- Average Freight
Definition: Average freight value recorded for sales rows.
Formula:
Avg Freight = AVERAGE(fact_sales[freight_value])

- Total Customers
Definition: Number of unique customers in the current filter context.
Formula:
Total Customers = DISTINCTCOUNT(dim_customer[customer_unique_id])

- Orders per Customer
Definition: Average number of unique orders per customer.
Formula:
Orders / Customer = Total Orders / Total Customers

- Revenue per Customer
Definition: Average revenue generated per unique customer.
Formula:
Revenue per Customer = Total Revenue / Total Customers

- Total Delayed Orders
Definition: Number of unique orders delivered after their estimated delivery date.
Formula:
Total Delayed Orders =
DISTINCTCOUNT(order_id)
where delayed_flag = 1

- Average Delivery Days
Definition: Average number of days between order purchase and actual customer delivery, calculated once per order.
Formula:
Avg Delivery Days =
average of delivery_days at unique order level

- Average Delay Days
Definition: Average number of days by which delayed orders exceeded the estimated delivery date, calculated once per delayed order.
Formula:
Avg Delay Days =
average of delay_days for orders where delayed_flag = 1

- Delayed Order Rate %
Definition: Percentage of eligible delivered orders that were delivered after the estimated delivery date.
Formula:
Delayed Order Rate % =
Delayed Orders / Orders with known delivery SLA

- % On-Time Delivery
Definition: Percentage of eligible orders delivered on or before their estimated delivery date.
Formula:
% On-Time Delivery =
On-Time Orders / Orders with known delivery SLA

- Regional Revenue Share %
Definition: Revenue in the selected geographical context compared with the total revenue available within the user's authorized RLS scope.
Formula:
Regional Revenue Share % =
Selected Revenue /
CALCULATE(
    [Total Revenue],
    REMOVEFILTERS(dim_customer)
)


Delivery SLA Definition

An order is considered:

- On-Time when actual delivery date <= estimated delivery date.
- Delayed when actual delivery date > estimated delivery date.
- SLA-eligible only when both actual and estimated delivery dates are available.

delivery_days = actual delivery date - order purchase date

delay_days = actual delivery date - estimated delivery date


RFM Segmentation

RFM segmentation is calculated at customer_unique_id level using:

- Recency – days since the customer's latest purchase,
- Frequency – number of unique orders,
- Monetary – total product value purchased.

Customers are assigned to the following segments:
Champion, At risk, Newbies, Regulars and Sleeping.


4. Functional Requirements

The report contains four main business pages:

Strategic Dashboard – Executive Director (3 pages):
- KPI Overview
- Map / geographical analysis with State -> City drill-down
- RFM customer segmentation

Operational Dashboard – Operations Manager (1 page):
- Operations

The strategic dashboard must support navigation through interactive buttons and bookmarks.

The Map page must provide geographical drill-down and a fallback state-level visual if map visuals are unavailable.

The Operational page must show delivery-time distribution, delivery bottlenecks, delayed-order trends and visually highlight delayed or anomalous records.

Order_Details and tooltip pages are supporting report pages and are not counted as main dashboard pages.

The report provides date filtering and metric selection where applicable.


5. Non-Functional & Technical Requirements

Security – Dynamic Row-Level Security:
Dynamic RLS must restrict Operations / Regional Managers to their assigned Brazilian state or states using customer_state.

The security mapping is stored in dim_security and the logged-in user is identified using USERPRINCIPALNAME().

The Executive Director must have unrestricted access to all states.

Advanced DAX Requirements:
- SUMX is used for row-by-row Total Revenue calculation.
- REMOVEFILTERS is used for revenue-share / company-total calculations while respecting RLS.
- SAMEPERIODLASTYEAR is used for previous-year comparison and YoY analysis.
- Time-intelligence calculations must use the active order-date relationship with dim_date.

Data Model:
Power BI must use the Mart Star Schema as the reporting semantic model.

Data Freshness:
Target refresh frequency is daily. In the local project environment, database and Power BI refreshes may be executed manually.

Performance:
The report should use the optimized Star Schema and efficient DAX measures with a target page load time below 3 seconds.


STEP 3 - Wireframing & Mockups

Low-fidelity wireframes were prepared before final dashboard implementation for:

- KPI Overview
- Map / Drill-down
- RFM Segmentation
- Operations Dashboard

The wireframes define the initial placement of KPI cards, filters, navigation and main/supporting visuals.

The final wireframe image is included separately in the Stage 3 deliverables.

STEP 4

Pattern 1 - Customer value is concentrated in the Regulars segment

The Regulars segment contains the largest number of customers and generates the highest total revenue. At the same time, Champions represent only a very small part of the customer base but generate much higher revenue per customer.

Hypothesis:
Most customers make only a limited number of repeat purchases, while a small group of highly engaged customers generates significantly more value per customer.

Business Action:
The company should focus retention and repeat-purchase campaigns on high-value Regulars and At-risk customers. Converting even a small proportion of these customers into Champions could increase customer lifetime value without relying only on new customer acquisition.


Pattern 2 - Delivery delays show visible peaks during periods of higher activity

Most deliveries are completed within the 6-10 day delivery-time bucket, while the overall delayed order rate is approximately 6.77%. The delayed-order trend also shows several visible peaks, particularly during late 2017 and early 2018.

Hypothesis:
Periods with higher order activity may put additional pressure on fulfillment and logistics capacity, which can contribute to an increase in delayed deliveries.

Business Action:
Operations should monitor delayed-order rate together with order volume and prepare additional logistics capacity before expected demand peaks. State-level monitoring can then be used to identify locations where operational intervention is required.

STEP 5

Three aggregated dashboard metrics were independently validated against raw SQL queries executed directly on the Mart layer:

- Total Revenue
- Total Orders
- Total Delayed Orders

The Power BI results matched the corresponding SQL query results.

Side-by-side Power BI vs SQL screenshots are stored separately as part of the Stage 3 validation evidence.